# TFP-BD: Encroachment with ResNet50 and MobileNetV3-Large (frame classifier, 70/20/10)

Puro frame dekhe model bole **Clear** na **Encroached** (FootpathVision paper er moto). Label ashe detector evaluation er
**same footpath zone rule** theke (annotation box), tai ground truth hubohu same, ar test hoy **same 1,605 frame** e.
Threshold validation e bachai (best kappa), test e ekbar. Shesh e 5 ta detector er sathe kappa, CI ar McNemar tulona.

**Input (Add Input):**
1. Bangladeshi Traffic Flow Dataset
2. Data Preparation (Sharmin) - `tfp_dataset`
3. **TFPBD_Final_Evaluation** er output (`alignment_cache.pkl`) - Add Input -> Your Work -> Notebooks
4. **TFPBD_Eval_Part2** er output (detector result, 7 model tulonar jonno) - Add Input -> Your Work -> Notebooks

**Settings:** GPU T4 x2 (ekta GPU use hobe), **Internet ON**.

Cell 1 chalie dekho sob input `OK` kina, tarpor **Save Version -> Save & Run All**. Somoy anumanik 45-75 min.
Shesh e Output e `enc_cls_results.zip` thakbe -> amake pathao.

In [ ]:
import os, re, glob, time, math, pickle, random, zipfile, warnings
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt
from matplotlib.path import Path
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
from scipy.stats import binomtest
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEV, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(no GPU)")
print("torch", torch.__version__, "| torchvision", torchvision.__version__)

WK   = "/kaggle/working"
OUTD = f"{WK}/enc_cls_results"
os.makedirs(OUTD, exist_ok=True)

CLASSES  = ["Car", "Bus", "Truck", "Motorcycle", "Bicycle", "Rickshaw", "Mini-Truck", "CNG", "Pedestrian"]
PED      = {8}
LOC_NAME = {1: "Arambag", 2: "Shapla Chattar", 3: "Abul Hotel", 4: "Bashabo"}
EXTS     = (".jpg", ".jpeg", ".png", ".bmp")
DETECTORS = ["YOLOv8n", "YOLO11n", "RT-DETR-L", "Faster R-CNN", "EfficientDet-D3"]

CLS_MODELS = ["ResNet50", "MobileNetV3-Large"]
IMG_W, IMG_H = 448, 336
BATCH = 32
EPOCHS = {"ResNet50": 10, "MobileNetV3-Large": 10}
LR = {"ResNet50": 3e-4, "MobileNetV3-Large": 1e-3}
NUM_WORKERS = 4
THRS = [round(x, 2) for x in np.arange(0.05, 0.96, 0.05)]

def safe(name):
    return re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")

BIG_DIRS = {"tfp_dataset", "Bangladeshi Traffic Flow Dataset", "images", "labels"}
F = {"ORIG": None, "SH": None, "ALIGN": None, "DET_FRAMES": None}
for root, dirs, files in os.walk("/kaggle/input"):
    if F["ORIG"] is None and "Bangladeshi Traffic Flow Dataset" in dirs:
        c = os.path.join(root, "Bangladeshi Traffic Flow Dataset")
        if os.path.isdir(os.path.join(c, "Annotated Images")):
            F["ORIG"] = c
    if F["SH"] is None and "tfp_dataset" in dirs:
        c = os.path.join(root, "tfp_dataset")
        if all(os.path.isdir(os.path.join(c, s, "images")) for s in ("train", "val", "test")):
            F["SH"] = c
    dirs[:] = [d for d in dirs if d not in BIG_DIRS]
    for f in files:
        p = os.path.join(root, f)
        if f == "alignment_cache.pkl":
            F["ALIGN"] = p
        elif f == "encroachment_test_per_frame.csv" and "eval_results_part2" in root:
            F["DET_FRAMES"] = p

ORIG, SH = F["ORIG"], F["SH"]
print("\n" + "=" * 70)
for k, v in F.items():
    tag = "OK " if v else ("MISSING" if k in ("ORIG", "SH") else "-- (optional)")
    print(f"{tag:14s} {k:12s} {v if v else ''}")
print("=" * 70)
assert ORIG and SH, "Add the Bangladeshi Traffic Flow Dataset and Sharmin's Data Preparation notebook with Add Input"
if not F["ALIGN"]:
    print("alignment_cache.pkl not found: the alignment is computed in Cell 2 (about 30 min). "
          "Add the TFPBD_Final_Evaluation output to skip this.")
if not F["DET_FRAMES"]:
    print("Detector results not found: the comparison with the 5 detectors is skipped. "
          "Add the TFPBD_Eval_Part2 output to include it.")

try:
    torchvision.models.resnet50(weights="IMAGENET1K_V2")
    torchvision.models.mobilenet_v3_large(weights="IMAGENET1K_V2")
    print("Pretrained weights OK (Internet ON)")
except Exception as e:
    raise RuntimeError("Could not download the pretrained weights - turn Internet ON in Settings. " + str(e)[:200])

In [ ]:
ZONES = {
    (1, "DL", 1): [[(0, .45), (.24, .45), (.235, .55), (.165, .74), (.065, 1), (0, 1)],
                   [(.68, .56), (.8, .61), (1, .74), (1, .92), (.85, .79), (.75, .72), (.66, .63)]],
    (1, "DL", 2): [[(0, .45), (.27, .45), (.25, .55), (.195, .74), (.105, 1), (0, 1)],
                   [(.68, .57), (.78, .61), (.95, .69), (.95, .92), (.85, .84), (.7, .75), (.66, .68)]],
    (1, "DL", 3): [[(0, 0), (.25, 0), (.24, .12), (.2, .24), (.12, .36), (.03, .45), (0, .47)],
                   [(.72, .1), (.85, .1), (1, .3), (1, .65), (.9, .5), (.78, .32), (.72, .2)]],
    (1, "DL", 4): [[(0, .45), (.24, .45), (.235, .55), (.165, .74), (.065, 1), (0, 1)],
                   [(.68, .53), (.8, .57), (1, .68), (1, .88), (.85, .75), (.75, .68), (.66, .6)]],
    (1, "DL", 5): [[(0, .15), (.12, .15), (.12, .3), (.08, .45), (.03, .6), (0, .65)],
                   [(.6, .28), (1, .28), (1, .42), (.62, .38)]],
    (1, "SL", 1): [[(0, .5), (.2, .44), (.33, .42), (.32, .52), (.29, .62), (.16, 1), (0, 1)]],
    (1, "SL", 2): [[(0, .5), (.2, .42), (.35, .4), (.34, .48), (.3, .6), (.12, 1), (0, 1)]],
    (1, "SL", 3): [[(0, .45), (.2, .38), (.35, .36), (.34, .48), (.3, .6), (.12, 1), (0, 1)]],
    (1, "SL", 4): [[(0, .5), (.2, .42), (.35, .4), (.34, .48), (.3, .6), (.12, 1), (0, 1)]],
    (1, "SL", 5): [[(0, .3), (.3, .3), (.32, .42), (.27, .6), (.2, .8), (.15, 1), (0, 1)]],
    (3, "DL", 1): [[(.55, .05), (.68, 0), (1, 0), (1, .5), (.76, .5), (.68, .4), (.6, .2)],
                   [(0, .05), (.2, .05), (.2, .2), (.12, .35), (0, .42)]],
    (3, "DL", 2): [[(.64, 0), (1, 0), (1, .55), (.9, .42), (.78, .27), (.7, .15)],
                   [(0, 0), (.2, 0), (.15, .2), (.1, .4), (.06, .6), (0, .7)]],
    (3, "DL", 3): [[(.6, 0), (1, 0), (1, .7), (.92, .55), (.8, .35), (.7, .2)],
                   [(0, 0), (.2, 0), (.13, .2), (.07, .4), (.03, .55), (0, .6)]],
    (3, "DL", 4): [[(.6, 0), (1, 0), (1, .45), (.85, .35), (.72, .2), (.62, .08)],
                   [(0, .05), (.22, .05), (.18, .25), (.12, .45), (.08, .6), (0, .65)]],
    (3, "DL", 5): [[(.68, 0), (1, 0), (1, 1), (.92, 1), (.8, .6), (.72, .3)],
                   [(0, .1), (.2, .1), (.15, .3), (.1, .5), (.05, .65), (0, .7)]],
    (3, "SL", 1): [[(.52, .2), (.62, .15), (1, .45), (1, 1), (.78, 1), (.68, .75), (.6, .55), (.54, .35)]],
    (3, "SL", 2): [[(.58, 0), (1, 0), (1, .72), (.9, .6), (.75, .35), (.62, .1)]],
    (3, "SL", 3): [[(.62, 0), (1, 0), (1, 1), (.92, 1), (.85, .6), (.75, .4), (.66, .2)]],
    (3, "SL", 4): [[(.62, 0), (1, 0), (1, .55), (.9, .4), (.75, .25), (.64, .12)]],
    (3, "SL", 5): [[(.58, 0), (1, 0), (1, .6), (.92, .62), (.8, .4), (.66, .2)]],
    (4, "DL", 1): [[(.55, .14), (.64, .24), (.8, .4), (.9, .49), (1, .56), (1, .7), (.9, .62), (.8, .54), (.64, .38), (.56, .22)],
                   [(0, .15), (.22, .15), (.2, .3), (.12, .45), (.05, .55), (0, .6)]],
    (4, "DL", 2): [[(.6, .4), (.7, .44), (.85, .5), (1, .56), (1, .72), (.85, .63), (.7, .55), (.6, .48)],
                   [(0, .4), (.2, .4), (.22, .5), (.18, .62), (.12, .75), (0, .85)]],
    (4, "DL", 3): [[(.55, .18), (.7, .32), (.85, .46), (1, .56), (1, .78), (.85, .64), (.75, .56), (.65, .5), (.58, .38), (.53, .25)],
                   [(0, .2), (.2, .2), (.25, .3), (.2, .45), (.12, .6), (.05, .75), (0, .8)]],
    (4, "DL", 4): [[(.58, .18), (.75, .25), (.9, .33), (1, .4), (1, .62), (.88, .52), (.75, .45), (.62, .35), (.56, .25)],
                   [(0, .15), (.22, .15), (.25, .25), (.2, .4), (.12, .5), (0, .55)]],
    (4, "DL", 5): [[(.58, .22), (.75, .32), (.9, .42), (1, .47), (1, .65), (.88, .58), (.75, .5), (.6, .35)],
                   [(0, .25), (.25, .25), (.28, .35), (.22, .55), (.15, .75), (.1, 1), (0, 1)]],
    (4, "SL", 1): [[(.48, .33), (.6, .42), (.75, .58), (.9, .72), (1, .78), (1, .92), (.88, .82), (.72, .68), (.58, .52), (.46, .4)]],
    (4, "SL", 2): [[(.48, .4), (.62, .5), (.75, .6), (.9, .72), (1, .78), (1, .95), (.88, .86), (.72, .74), (.6, .62), (.47, .48)]],
    (4, "SL", 3): [[(.5, .22), (.62, .28), (.72, .36), (.86, .52), (.95, .64), (1, .7), (1, .82), (.92, .76), (.84, .66), (.72, .54), (.6, .4), (.49, .3)]],
    (4, "SL", 4): [[(.53, .33), (.7, .45), (.85, .56), (1, .66), (1, 1), (.85, .9), (.75, .8), (.6, .6), (.52, .42)]],
    (4, "SL", 5): [[(0, .4), (.25, .4), (.35, .45), (.33, .6), (.3, .8), (.28, 1), (0, 1)]],
}

def key_of(p):
    m = re.search(r"Location (\d)", p)
    t = re.match(r"(\d)-LOC", os.path.basename(os.path.dirname(p)))
    lane = "DL" if "Double" in p else ("SL" if "Single" in p else None)
    return (int(m.group(1)), lane, int(t.group(1))) if (m and t and lane) else None

def stem_of(p):
    return os.path.splitext(os.path.basename(p))[0]

key_of_stem, folder_frames = {}, {}
for p in glob.glob(f"{ORIG}/Annotated Images/**/*", recursive=True):
    if p.lower().endswith(EXTS):
        k = key_of(p)
        key_of_stem[stem_of(p)] = k
        if k in ZONES:
            folder_frames.setdefault(k, []).append(p)
print(f"Camera folder: {len(folder_frames)} / 30")

W0, H0 = 640, 480
CORN = np.float32([[0, 0], [W0, 0], [W0, H0], [0, H0]]).reshape(-1, 1, 2)
MIN_INL, MIN_SCORE = 30, 0.15

def load(p):
    img = cv2.imread(p)
    if img is None:
        return None
    g = cv2.cvtColor(cv2.resize(img, (W0, H0)), cv2.COLOR_BGR2GRAY)
    gs = cv2.createCLAHE(2.0, (8, 8)).apply(cv2.resize(g, (480, 360)))
    kp, d = cv2.SIFT_create(2000).detectAndCompute(gs, None)
    if d is None or len(kp) < 20:
        return None
    return {"g": g, "pts": np.float32([k.pt for k in kp]) * (W0 / 480), "d": d}

def sane(H):
    q = cv2.perspectiveTransform(CORN, H)
    area = cv2.contourArea(q) / (W0 * H0)
    return 0.3 < area < 3.0 and cv2.isContourConvex(q.astype(np.float32))

def homog(fa, fb):
    if fa is None or fb is None:
        return None, 0
    kn = cv2.FlannBasedMatcher(dict(algorithm=1, trees=5), dict(checks=32)).knnMatch(fa["d"], fb["d"], k=2)
    good = [x[0] for x in kn if len(x) == 2 and x[0].distance < 0.75 * x[1].distance]
    if len(good) < MIN_INL:
        return None, len(good)
    A = fa["pts"][[x.queryIdx for x in good]]
    B = fb["pts"][[x.trainIdx for x in good]]
    H, inl = cv2.findHomography(A, B, cv2.USAC_MAGSAC, 3.0, maxIters=5000, confidence=0.999)
    if H is None or inl is None:
        return None, 0
    n = int(inl.sum())
    if n < MIN_INL or not sane(H):
        return None, n
    return H / H[2, 2], n

def _grad(g):
    g = cv2.GaussianBlur(g.astype(np.float32), (0, 0), 1.5)
    return cv2.Sobel(g, cv2.CV_32F, 1, 0), cv2.Sobel(g, cv2.CV_32F, 0, 1)

def score(fref, f, H):
    wa = cv2.warpPerspective(fref["g"], H, (W0, H0))
    wm = cv2.warpPerspective(np.full((H0, W0), 255, np.uint8), H, (W0, H0), flags=cv2.INTER_NEAREST)
    valid = cv2.erode(wm, np.ones((9, 9), np.uint8)) > 0
    if valid.mean() < 0.15:
        return 0.0
    ax, ay = _grad(wa)
    bx, by = _grad(f["g"])
    num = (ax * bx + ay * by)[valid].sum()
    den = np.sqrt((ax ** 2 + ay ** 2)[valid].sum() * (bx ** 2 + by ** 2)[valid].sum()) + 1e-6
    return float(num / den)

def zshift(k, H):
    d = []
    for poly in ZONES[k]:
        pts = np.float32([[x * W0, y * H0] for x, y in poly]).reshape(-1, 1, 2)
        q = cv2.perspectiveTransform(pts, H).reshape(-1, 2) - pts.reshape(-1, 2)
        d += list(np.hypot(q[:, 0] / W0, q[:, 1] / H0))
    return float(np.median(d) * 100)

def fnum(p):
    m = re.search(r"frame(\d+)", os.path.basename(p))
    return int(m.group(1)) if m else 0

def align_folder(k):
    fl = sorted(folder_frames[k])
    ref = fl[0]
    fr = load(ref)
    out = {stem_of(ref): (k, np.eye(3), "ref", 0.0, 1.0)}
    last = (fr, np.eye(3))
    for p in sorted(fl[1:], key=fnum):
        f = load(p)
        best = (None, "fail", 0.0)
        H, _ = homog(fr, f)
        if H is not None:
            s = score(fr, f, H)
            best = (H, "direct", s) if s >= MIN_SCORE else (None, "fail", s)
        if best[0] is None and last[0] is not None and last[0] is not fr:
            Hc, _ = homog(last[0], f)
            if Hc is not None:
                H2 = Hc @ last[1]
                H2 = H2 / H2[2, 2]
                if sane(H2):
                    s2 = score(fr, f, H2)
                    if s2 >= MIN_SCORE:
                        best = (H2, "chain", s2)
        H, how, s = best
        if H is None:
            out[stem_of(p)] = (k, None, "fail", np.nan, s)
        else:
            out[stem_of(p)] = (k, H, how, zshift(k, H), s)
            last = (f, H)
    return k, out


if F["ALIGN"]:
    AL = pickle.load(open(F["ALIGN"], "rb"))
    print(f"Alignment cache loaded: {F['ALIGN']} ({len(AL)} frames)")
else:
    t0 = time.time()
    AL = {}
    with ThreadPoolExecutor(max_workers=4) as ex:
        for i, (k, out) in enumerate(ex.map(align_folder, sorted(folder_frames))):
            AL.update(out)
            if (i + 1) % 5 == 0:
                print(f"  alignment: {i + 1}/{len(folder_frames)} folders ({time.time() - t0:.0f}s)")
    pickle.dump(AL, open(f"{WK}/alignment_cache.pkl", "wb"))
    print(f"Alignment done in {time.time() - t0:.0f}s, saved alignment_cache.pkl")

al = pd.DataFrame([{"stem": s, "key": str(v[0]), "how": v[2], "shift": v[3], "score": v[4]} for s, v in AL.items()])
al.to_csv(f"{OUTD}/frame_alignment.csv", index=False)
al["fail"] = al["how"] == "fail"
print(f"Total {len(al)} frames | aligned {100 - al['fail'].mean() * 100:.1f}%")
bad = al.groupby("key")["fail"].mean()
print("Camera folders where alignment failed on more than 5% of frames:")
print((bad[bad > 0.05] * 100).round(1).to_string())

ZCACHE = {}
def zone_polys(stem):
    if stem in ZCACHE:
        return ZCACHE[stem]
    v = AL.get(stem)
    res = None
    if v is not None and v[1] is not None:
        res = []
        for poly in ZONES[v[0]]:
            pts = np.float32([[x * W0, y * H0] for x, y in poly]).reshape(-1, 1, 2)
            res.append(Path(cv2.perspectiveTransform(pts, v[1]).reshape(-1, 2) / [W0, H0]))
    ZCACHE[stem] = res
    return res

In [ ]:
def list_split(s):
    d = f"{SH}/{s}/images"
    return sorted(os.path.join(d, f) for f in os.listdir(d) if f.lower().endswith(EXTS))

def boxes_from_label(lp):
    out = []
    if os.path.exists(lp):
        with open(lp) as fh:
            for line in fh:
                q = line.split()
                if len(q) >= 5:
                    c, xc, yc, w, h = int(q[0]), *map(float, q[1:5])
                    out.append((c, [xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2]))
    return out

PATHS = {s: list_split(s) for s in ("train", "val", "test")}
GT = {s: {stem_of(p): boxes_from_label(f"{SH}/{s}/labels/{stem_of(p)}.txt") for p in PATHS[s]} for s in ("val", "test")}
print({s: len(v) for s, v in PATHS.items()})

def foot(b):
    x0, y0, x1, y1 = b
    return (min(max((x0 + x1) / 2, 0.001), 0.999), min(max(y1, 0.001), 0.995))

def any_in_zone(boxes, stem):
    z = zone_polys(stem)
    if z is None:
        return False
    return any(any(P.contains_point(foot(b)) for P in z) for c, b in boxes if c not in PED)

def enc_pred(dets, stem, thr):
    return any_in_zone([(c, b) for c, b, s in dets if s >= thr], stem)

def enc_frames(split):
    return sorted(s for s in GT[split] if key_of_stem.get(s) in ZONES and zone_polys(s) is not None)

ENC = {s: enc_frames(s) for s in ("val", "test")}
GT_ENC = {s: {st: any_in_zone(GT[s][st], st) for st in ENC[s]} for s in ("val", "test")}
print(f"Encroachment frames (Locations 1/3/4, zone aligned): val {len(ENC['val'])} | test {len(ENC['test'])}")
print(f"Encroached in the test annotations: {np.mean(list(GT_ENC['test'].values())) * 100:.1f}%")

def summarize(p, g):
    p, g = np.asarray(p, dtype=bool), np.asarray(g, dtype=bool)
    n = len(p)
    tp, fp = int((p & g).sum()), int((p & ~g).sum())
    fn, tn = int((~p & g).sum()), int((~p & ~g).sum())
    po, pm, pg = (tp + tn) / n, (tp + fp) / n, (tp + fn) / n
    pc = pm * pg + (1 - pm) * (1 - pg)
    pr, rc = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return {"n": n, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "model_enc_%": pm * 100, "gt_enc_%": pg * 100, "agree_%": po * 100,
            "kappa": (po - pc) / (1 - pc) if pc < 1 else float("nan"),
            "recall_%": rc * 100, "precision_%": pr * 100,
            "f1_%": 2 * pr * rc / max(pr + rc, 1e-9) * 100}

GT["train"] = {stem_of(p): boxes_from_label(f"{SH}/train/labels/{stem_of(p)}.txt") for p in PATHS["train"]}
ENC["train"] = enc_frames("train")
GT_ENC["train"] = {st: any_in_zone(GT["train"][st], st) for st in ENC["train"]}
PATH_OF = {s: {stem_of(p): p for p in PATHS[s]} for s in ("train", "val", "test")}
print()
for s in ("train", "val", "test"):
    print(f"{s:5s}: {len(ENC[s]):6d} frames | Encroached {np.mean(list(GT_ENC[s].values())) * 100:.1f}%")
SAME_FRAMES = len(ENC["val"]) == 3279 and len(ENC["test"]) == 1605
if F["ALIGN"]:
    assert SAME_FRAMES, "Frame counts differ from the detector evaluation (val 3,279 / test 1,605)"
elif not SAME_FRAMES:
    print("Warning: frame counts differ from the detector evaluation, the detector comparison will be skipped")

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

class FrameDS(Dataset):
    def __init__(self, split, train):
        self.items = [(PATH_OF[split][s], int(GT_ENC[split][s])) for s in ENC[split]]
        aug = [T.ColorJitter(0.3, 0.3, 0.2),
               T.RandomAffine(degrees=0, translate=(0.02, 0.02), scale=(0.95, 1.05))] if train else []
        self.tf = T.Compose([T.Resize((IMG_H, IMG_W))] + aug + [T.ToTensor(), T.Normalize(MEAN, STD)])
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        p, y = self.items[i]
        return self.tf(Image.open(p).convert("RGB")), y

def make_loader(split, train):
    return DataLoader(FrameDS(split, train), batch_size=BATCH, shuffle=train, num_workers=NUM_WORKERS,
                      pin_memory=DEV.type == "cuda", persistent_workers=NUM_WORKERS > 0, drop_last=train)

LOADERS = {s: make_loader(s, s == "train") for s in ("train", "val", "test")}
GT_ARR = {s: np.array([GT_ENC[s][st] for st in ENC[s]], dtype=bool) for s in ("val", "test")}

def build_model(name):
    if name == "ResNet50":
        m = torchvision.models.resnet50(weights="IMAGENET1K_V2")
        m.fc = nn.Linear(m.fc.in_features, 2)
    elif name == "MobileNetV3-Large":
        m = torchvision.models.mobilenet_v3_large(weights="IMAGENET1K_V2")
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, 2)
    else:
        raise ValueError(name)
    return m.to(DEV)

def make_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=DEV.type == "cuda")
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=DEV.type == "cuda")

@torch.no_grad()
def predict_probs(model, split, crit=None):
    model.eval()
    out, loss_sum, n = [], 0.0, 0
    for x, y in LOADERS[split]:
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEV.type == "cuda"):
            logits = model(x.to(DEV, non_blocking=True))
        if crit is not None:
            loss_sum += crit(logits.float(), y.to(DEV)).item() * len(y)
            n += len(y)
        out.append(torch.softmax(logits.float(), 1)[:, 1].cpu().numpy())
    prob = np.concatenate(out)
    return (prob, loss_sum / max(n, 1)) if crit is not None else prob

def sweep(prob, gt):
    return pd.DataFrame([{"thr": t, **summarize(prob >= t, gt)} for t in THRS])

def best_row(sw):
    return sw.loc[sw["kappa"].fillna(-1).idxmax()]

def train_model(name):
    model = build_model(name)
    epochs, steps = EPOCHS[name], len(LOADERS["train"])
    opt = torch.optim.AdamW(model.parameters(), lr=LR[name], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR[name], total_steps=epochs * steps, pct_start=0.15)
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)
    scaler = make_scaler()
    best, hist, t0 = -1.0, [], time.time()
    ckpt = f"{WK}/enc_{safe(name)}_best.pt"
    for ep in range(1, epochs + 1):
        model.train()
        run_loss, correct, n = 0.0, 0, 0
        for x, y in LOADERS["train"]:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=DEV.type == "cuda"):
                logits = model(x)
                loss = crit(logits, y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            run_loss += loss.item() * len(y)
            correct += int((logits.argmax(1) == y).sum().item())
            n += len(y)
        pv, val_loss = predict_probs(model, "val", crit)
        r = best_row(sweep(pv, GT_ARR["val"]))
        hist.append({"epoch": ep, "train_loss": run_loss / n, "train_acc_%": correct / n * 100,
                     "val_loss": val_loss, "val_acc_%": float(((pv >= 0.5) == GT_ARR["val"]).mean() * 100),
                     "val_best_thr": float(r["thr"]), "val_kappa": float(r["kappa"]),
                     "val_agree_%": float(r["agree_%"]), "minutes": (time.time() - t0) / 60})
        tag = ""
        if r["kappa"] > best:
            best = float(r["kappa"])
            torch.save(model.state_dict(), ckpt)
            tag = "  <- best"
        h = hist[-1]
        print(f"[{name}] epoch {ep}/{epochs} | train loss {h['train_loss']:.4f} acc {h['train_acc_%']:.1f}% | "
              f"val loss {h['val_loss']:.4f} acc {h['val_acc_%']:.1f}% | val kappa {r['kappa']:.3f} at thr {r['thr']:.2f} | "
              f"{h['minutes']:.1f} min{tag}")
    pd.DataFrame(hist).to_csv(f"{OUTD}/history_enc_{safe(name)}.csv", index=False)
    model.load_state_dict(torch.load(ckpt, map_location=DEV))
    return model, hist, (time.time() - t0) / 60

xb, yb = next(iter(LOADERS["val"]))
print("Batch:", tuple(xb.shape), "| labels:", yb[:16].tolist())
for name in CLS_MODELS:
    m = build_model(name)
    m.train()
    with torch.autocast("cuda", dtype=torch.float16, enabled=DEV.type == "cuda"):
        out = m(xb[:4].to(DEV))
    out.float().sum().backward()
    print(f"{name}: forward and backward OK, output {tuple(out.shape)}")
    del m, out
if DEV.type == "cuda":
    torch.cuda.empty_cache()
print("Functions ready")

In [ ]:
MODELS = {}
MODELS["ResNet50"] = train_model("ResNet50")

In [ ]:
MODELS["MobileNetV3-Large"] = train_model("MobileNetV3-Large")

In [ ]:
@torch.no_grad()
def speed_ms(model, n=200):
    model.eval()
    ds = FrameDS("test", False)
    xs = [ds[i][0].unsqueeze(0).to(DEV) for i in range(min(n, len(ds)))]
    for x in xs[:20]:
        model(x)
    if DEV.type == "cuda":
        torch.cuda.synchronize()
    t = time.time()
    for x in xs:
        model(x)
    if DEV.type == "cuda":
        torch.cuda.synchronize()
    return (time.time() - t) / len(xs) * 1000

test_tab = pd.DataFrame({"stem": ENC["test"]})
test_tab["loc"] = [key_of_stem[s][0] for s in ENC["test"]]
test_tab["night"] = [key_of_stem[s][2] == 5 for s in ENC["test"]]
test_tab["gt"] = GT_ARR["test"]

def scores(name, frame):
    row = summarize(frame[name], frame["gt"])
    for L in (1, 3, 4):
        s = frame[frame["loc"] == L]
        row[f"kappa_{LOC_NAME[L]}"] = summarize(s[name], s["gt"])["kappa"]
    row["kappa_day"] = summarize(frame.loc[~frame["night"], name], frame.loc[~frame["night"], "gt"])["kappa"]
    row["kappa_night"] = summarize(frame.loc[frame["night"], name], frame.loc[frame["night"], "gt"])["kappa"]
    return row

rows, sweeps = [], {}
for name, (model, hist, train_min) in MODELS.items():
    sw = sweep(predict_probs(model, "val"), GT_ARR["val"])
    sw.to_csv(f"{OUTD}/enc_val_sweep_{safe(name)}.csv", index=False)
    sweeps[name] = sw
    thr = float(best_row(sw)["thr"])
    prob = predict_probs(model, "test")
    test_tab[f"{name}_prob"] = prob
    test_tab[name] = prob >= thr
    rows.append({"model": name, "val_best_thr": thr, **scores(name, test_tab),
                 "params_M": round(sum(q.numel() for q in model.parameters()) / 1e6, 2),
                 "ms_per_frame_bs1": round(speed_ms(model), 2), "train_minutes": round(train_min, 1),
                 "best_epoch": max(hist, key=lambda h: h["val_kappa"])["epoch"]})

enc_cls = pd.DataFrame(rows)
enc_cls.to_csv(f"{OUTD}/encroachment_test_classifiers.csv", index=False)
test_tab.to_csv(f"{OUTD}/encroachment_test_per_frame_classifiers.csv", index=False)
print("=" * 70 + f"\nENCROACHMENT, FRAME CLASSIFIERS | test {len(test_tab)} frames | "
      f"annotated Encroached {test_tab['gt'].mean() * 100:.1f}%\n" + "=" * 70)
cols = ["model", "val_best_thr", "agree_%", "kappa", "f1_%", "recall_%", "precision_%", "model_enc_%",
        "kappa_Arambag", "kappa_Abul Hotel", "kappa_Bashabo", "kappa_day", "kappa_night",
        "params_M", "ms_per_frame_bs1", "best_epoch"]
print(enc_cls[cols].round(3).to_string(index=False))

NAMES = list(MODELS)
if F["DET_FRAMES"] and SAME_FRAMES:
    det = pd.read_csv(F["DET_FRAMES"])
    same = (len(det) == len(test_tab) and (det["stem"].astype(str).values == test_tab["stem"].values).all()
            and (det["gt"].astype(bool).values == test_tab["gt"].values).all())
    if same:
        for c in DETECTORS:
            test_tab[c] = det[c].astype(bool).values
        NAMES = DETECTORS + NAMES
        print("\nDetector results matched frame by frame (same stems, same ground truth)")
    else:
        print("\nDetector file does not match these test frames: comparison skipped")

rng = np.random.default_rng(42)
ie = rng.integers(0, len(test_tab), (1000, len(test_tab)))
all_rows = []
for name in NAMES:
    p, g = test_tab[name].to_numpy(dtype=bool), test_tab["gt"].to_numpy(dtype=bool)
    ks = [summarize(p[ix], g[ix])["kappa"] for ix in ie]
    R = summarize(p, g)
    all_rows.append({"model": name, "type": "classifier" if name in MODELS else "detector",
                     "kappa": R["kappa"], "kappa_CI_low": np.nanpercentile(ks, 2.5),
                     "kappa_CI_high": np.nanpercentile(ks, 97.5), "agree_%": R["agree_%"], "f1_%": R["f1_%"],
                     "recall_%": R["recall_%"], "precision_%": R["precision_%"]})
all_df = pd.DataFrame(all_rows)
all_df.to_csv(f"{OUTD}/encroachment_test_all_models.csv", index=False)
print("\nAll models on the same test frames (kappa with bootstrap 95% CI, 1000 resamples, seed 42):")
print(all_df.round(3).to_string(index=False))

mc = []
for i in range(len(NAMES)):
    for j in range(i + 1, len(NAMES)):
        a = test_tab[NAMES[i]] == test_tab["gt"]
        b = test_tab[NAMES[j]] == test_tab["gt"]
        n01, n10 = int((a & ~b).sum()), int((~a & b).sum())
        p = binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
        mc.append({"model_A": NAMES[i], "model_B": NAMES[j], "only_A_correct": n01, "only_B_correct": n10,
                   "p_value": p, "significant_(p<0.05)": p < 0.05})
mc_df = pd.DataFrame(mc)
mc_df.to_csv(f"{OUTD}/encroachment_mcnemar_all_models.csv", index=False)
print("\nMcNemar (pairs with at least one classifier):")
print(mc_df[mc_df["model_A"].isin(MODELS) | mc_df["model_B"].isin(MODELS)].round(4).to_string(index=False))

from sklearn.metrics import classification_report
rep_rows = []
for name in NAMES:
    g, p = test_tab["gt"].astype(int), test_tab[name].astype(int)
    rep = classification_report(g, p, labels=[0, 1], target_names=["Clear", "Encroached"], digits=4,
                                output_dict=True, zero_division=0)
    for k, v in rep.items():
        rep_rows.append({"model": name, "row": k, **v} if isinstance(v, dict) else {"model": name, "row": k, "f1-score": v})
    if name in MODELS:
        print(f"\nClassification report, {name} (test, {len(test_tab)} frames):")
        print(classification_report(g, p, labels=[0, 1], target_names=["Clear", "Encroached"], digits=4, zero_division=0))
pd.DataFrame(rep_rows).to_csv(f"{OUTD}/classification_report_all_models.csv", index=False)

fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
for name, (model, hist, _) in MODELS.items():
    h = pd.DataFrame(hist)
    col = ax[0].plot(h["epoch"], h["train_loss"], marker="o", label=f"{name} train")[0].get_color()
    ax[0].plot(h["epoch"], h["val_loss"], marker="s", ls="--", color=col, label=f"{name} validation")
    ax[1].plot(h["epoch"], h["train_acc_%"], marker="o", color=col, label=f"{name} train")
    ax[1].plot(h["epoch"], h["val_acc_%"], marker="s", ls="--", color=col, label=f"{name} validation")
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Cross-entropy loss"); ax[0].set_title("Loss curve"); ax[0].legend()
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Accuracy (%)"); ax[1].set_title("Accuracy curve"); ax[1].legend()
plt.tight_layout()
plt.savefig(f"{OUTD}/enc_classifiers_loss_accuracy_curves.png", dpi=110, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(1, len(MODELS), figsize=(6 * len(MODELS), 5))
for a, name in zip(np.atleast_1d(ax), MODELS):
    g, p = test_tab["gt"].to_numpy(dtype=bool), test_tab[name].to_numpy(dtype=bool)
    cm = np.array([[(g & p).sum(), (g & ~p).sum()], [(~g & p).sum(), (~g & ~p).sum()]])
    a.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            a.text(j, i, f"{cm[i, j]}\n{cm[i, j] / cm[i].sum() * 100:.1f}%", ha="center", va="center", fontsize=13,
                   color="white" if cm[i, j] > cm.max() / 2 else "black")
    a.set_xticks([0, 1]); a.set_xticklabels(["Encroached", "Clear"]); a.set_xlabel("Model prediction")
    a.set_yticks([0, 1]); a.set_yticklabels(["Encroached", "Clear"]); a.set_ylabel("Annotation")
    a.set_title(f"{name} (test)")
plt.tight_layout()
plt.savefig(f"{OUTD}/enc_classifiers_confusion_matrices.png", dpi=110, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(19, 5))
for name, sw in sweeps.items():
    axes[0].plot(sw["thr"], sw["kappa"], marker="o", label=name)
axes[0].set_xlabel("Probability threshold (Encroached)")
axes[0].set_ylabel("Cohen's kappa (validation)")
axes[0].set_title("Validation threshold sweep")
axes[0].legend()
o = all_df.sort_values("kappa")
axes[1].errorbar(o["kappa"], range(len(o)), xerr=[o["kappa"] - o["kappa_CI_low"], o["kappa_CI_high"] - o["kappa"]],
                 fmt="o", capsize=4)
axes[1].set_yticks(range(len(o)))
axes[1].set_yticklabels(o["model"])
axes[1].set_xlabel("Cohen's kappa on test (95% CI)")
axes[1].set_title(f"Test, {len(test_tab)} frames")
cms = [[int((test_tab[n] & test_tab["gt"]).sum()), int((~test_tab[n] & test_tab["gt"]).sum()),
        int((test_tab[n] & ~test_tab["gt"]).sum()), int((~test_tab[n] & ~test_tab["gt"]).sum())] for n in MODELS]
axes[2].axis("off")
tb = axes[2].table(cellText=[[n] + c for n, c in zip(MODELS, cms)], colLabels=["Model", "TP", "FN", "FP", "TN"],
                   loc="center")
tb.auto_set_font_size(False)
tb.set_fontsize(11)
tb.scale(1, 2)
axes[2].set_title("Test confusion counts")
plt.tight_layout()
plt.savefig(f"{OUTD}/enc_classifiers_summary.png", dpi=110, bbox_inches="tight")
plt.show()

zp = f"{WK}/enc_cls_results.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(OUTD)):
        z.write(os.path.join(OUTD, f), f)
print("\nSaved:", zp)